# uq_setup
UQ experiment setup and execution notebook.

* gridkit installed in: `/home/isatkaus/gridkit`

#### workflow
1. run imports cell
2. run runner setup cell
3. run ONE case config cell (hawaii or illinois), then proceed with shared cells
4. create meta.yml, generate samples, create run dirs, run all, collect

#### references
* `/home/isatkaus/gridkit/uq-usecase/cases/hawaii.md` — hawaii UQ docs
* `/home/isatkaus/gridkit/uq-usecase/cases/illinois.md` — illinois UQ docs
* `/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py` — sampling + run utilities

In [4]:
import pandas as pd
import numpy as np
import os
import sys
import json
import glob
import shutil
import datetime
import subprocess
import yaml

import plotly.express as px
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.options.plotting.backend = "plotly"

# === GridKit paths ===
GRIDKIT_REPO_ROOT = os.path.expanduser("~/gridkit")
GRIDKIT_BUILD_DIR = os.path.join(GRIDKIT_REPO_ROOT, "build")
GRIDKIT_PY_UTILS = os.path.join(GRIDKIT_REPO_ROOT, "uq-usecase/py-utils")
if GRIDKIT_PY_UTILS not in sys.path:
    sys.path.insert(0, GRIDKIT_PY_UTILS)

print(f"GRIDKIT_REPO_ROOT: {GRIDKIT_REPO_ROOT}")
print(f"GRIDKIT_BUILD_DIR: {GRIDKIT_BUILD_DIR}")
print(f"GRIDKIT_PY_UTILS:  {GRIDKIT_PY_UTILS}")

GRIDKIT_REPO_ROOT: /home/isatkaus/gridkit
GRIDKIT_BUILD_DIR: /home/isatkaus/gridkit/build
GRIDKIT_PY_UTILS:  /home/isatkaus/gridkit/uq-usecase/py-utils


## runner setup

In [5]:
build_dir = GRIDKIT_BUILD_DIR
runner = os.path.join(build_dir, "application/PhasorDynamics/DynamicSimulation")
os.environ["PATH"] = os.path.dirname(runner) + os.pathsep + os.environ["PATH"]
print(f"runner: {runner}")
print(f"exists: {os.path.exists(runner)}")

runner: /home/isatkaus/gridkit/build/application/PhasorDynamics/DynamicSimulation
exists: True


In [6]:
!which DynamicSimulation

~/gridkit/build/application/PhasorDynamics/DynamicSimulation


## existing UQ runs

Scans all `meta.yml` files under `gridkit-runs/` and displays a summary table.
Re-run this cell at any time to refresh.


In [7]:
# === Existing UQ runs: scan meta.yml files and display summary table ===
# Columns:
#   n_samples   -- total planned samples (from meta.yml)
#   n_parquets  -- collected result files actually on disk (run_NNN.parquet for per_run,
#                  or n_samples if results.parquet exists for stacked; NA if output path missing)
#   complete    -- n_parquets / n_samples as a percentage
#   Genrou_H_ids -- comma-separated Genrou device IDs whose H parameter is perturbed
#                   (these are the UQ input parameters, one entry per sampled dimension)

_RUNS_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs"

_rows = []
for _meta_path in sorted(glob.glob(os.path.join(_RUNS_ROOT, "*/meta.yml"))):
    _run_name = os.path.basename(os.path.dirname(_meta_path))
    try:
        with open(_meta_path) as _f:
            _m = yaml.safe_load(_f)
    except Exception as _e:
        _rows.append({"run": _run_name, "error": str(_e)})
        continue

    _s = _m.get("sampling", {})
    _params = _s.get("params", [])
    _genrou_ids = ", ".join(p.get("id", "?") for p in _params)

    # count collected parquet files on disk
    _serialize = _m.get("serialize_mode", "unknown")
    _run_dir = os.path.dirname(_meta_path)
    _n_total = _s.get("n_samples", None)

    if _serialize == "per_run":
        _runs_path = os.path.join(_run_dir, "runs")
        if os.path.isdir(_runs_path):
            _n_done = len(glob.glob(os.path.join(_runs_path, "run_*.parquet")))
        else:
            _n_done = None  # runs/ dir not yet created
    elif _serialize == "stacked":
        _results_path = os.path.join(_run_dir, "results.parquet")
        if os.path.exists(_results_path):
            _n_done = _n_total  # single file = all collected
        else:
            _n_done = None
    else:
        _n_done = None

    _pct = (
        f"{_n_done / _n_total * 100:.0f}%" if _n_done is not None and _n_total else "NA"
    )

    _rows.append(
        {
            "run": _run_name,
            "case": _m.get("case", "?"),
            "created": _m.get("created", "?")[:10],
            "n_samples": int(_n_total) if _n_total is not None else "NA",
            "n_parquets": int(_n_done) if _n_done is not None else "NA",
            "complete": _pct,
            "method": _s.get("method", "?"),
            "dist": _s.get("dist_type", "?"),
            "serialize": _serialize,
            "Genrou_H_ids": _genrou_ids,
        }
    )

_runs_df = pd.DataFrame(_rows)
_runs_df.sort_values("run", ascending=True).reset_index(drop=True)

,run,case,created,n_samples,n_parquets,complete,method,dist,serialize,Genrou_H_ids
0,hawaii-ian-csv,Hawaii,2026-09-21,1000,1000,100%,from_csv,from_csv,per_run,"2_1, 2_2, 2_3, 2_4, 23_1, 23_10, 23_2, 23_3, 2..."
1,hawaii-v1,Hawaii,2026-06-11,NA,NA,NA,?,?,unknown,
2,hawaii-v10,Hawaii,2026-08-11,4000,4000,100%,lhs,uniform,per_run,"35_4, 23_9, 26_1, 27_2"
3,hawaii-v2,Hawaii,2026-06-11,NA,1000,NA,?,?,per_run,
4,hawaii-v3,Hawaii,2026-07-21,1000,1000,100%,lhs,normal,per_run,"2_1, 23_1, 34_1, 35_1"
...,...,...,...,...,...,...,...,...,...,...
11,illinois-ian-csv,Illinois,2026-09-22,1025,NA,NA,from_csv,from_csv,per_run,"49_1, 50_1, 51_1, 52_1, 53_1, 65_1, 67_1, 68_1..."
12,illinois-ian-csv-delete,Illinois,2026-09-22,1025,1025,100%,from_csv,from_csv,per_run,"49_1, 50_1, 51_1, 52_1, 53_1, 65_1, 67_1, 68_1..."
13,illinois-v1,Illinois,2026-07-21,1000,1000,100%,lhs,normal,per_run,"126_1, 135_1, 115_1, 189_1"
14,illinois-v2,Illinois,2026-07-21,4000,4000,100%,lhs,normal,per_run,"126_1, 135_1, 115_1, 189_1"


## case config
Run ONE of the setup cells below:

> hawaii setup

> illinoi setup

then proceed with the shared cells (meta, samples, run, collect).

### monitored variables

`MONITORS_BY_CLASS` in each config cell controls which signals are recorded per device class.
`make_run_dir` **overwrites** the `mon` array in every matching bus/device entry of the copied case JSON with these values — the base case values (`Vr`/`Vi` by default) are replaced.

| Class | Variable | Description |
|-------|----------|-------------|
| `bus` | `Vm` | voltage magnitude (pu) |
| `bus` | `Va` | voltage angle (rad) |
| `bus` | `Vr` | voltage real part (pu) — base case default |
| `bus` | `Vi` | voltage imaginary part (pu) — base case default |
| `genrou` | `delta` | rotor angle (rad) |
| `genrou` | `omega` | speed deviation from synchronous (pu); steady-state ~0 |

Current choice: **polar** (`Vm`, `Va`) instead of the base case rectangular (`Vr`, `Vi`).


### hawaii setup

In [19]:
# === UQ config (Hawaii) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "hawaii.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "hawaii.solver.json")

# -----------------------------------------------------------------------
# STEP 1: select ONE run root below (comment out all others)
# -----------------------------------------------------------------------
# UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v5"   # v5: 16K, normal std=12%, bus-diverse 4-gen (reference)
# UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v6"   # v6: 4K, uniform +-20%, smallest H
# UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v7"   # v7: 4K, uniform +-20%, farthest from fault
# UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v8"   # v8: 4K, uniform +-20%, smallest mva
# UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v9"   # v9: 4K, uniform +-20%, largest H*mva (positive control)
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v10"  # v10: 4K, uniform +-20%, small p0 / lightly dispatched (run after reviewing v9)

# "per_run" (default): one run_NNN.parquet per run under runs/ -- use for N>=100
# "stacked": all runs in one results.parquet -- only for small N (<500)
SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 16000  # v5 only; overridden inside each PARAM_SPECS block for v6-v10
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides ---
SOLVER_OVERRIDES = None

# -----------------------------------------------------------------------
# STEP 2: uncomment the PARAM_SPECS block that matches the run root above
#
# Sampling range rationale:
#   v5 used normal std=12% -- matches "utility database" epistemic uncertainty
#   v6-v10 use uniform +-20% -- wider but physically grounded:
#     +-2-5%  : manufacturer nameplate (well-measured rotor geometry)
#     +-10-15%: NERC/IEEE standard assumption for database values
#     +-20%   : "fairly uncertain database value", defensible prior for calibration
#     +-50%+  : stress-test / identifiability bound, not a realistic calibration prior
#
# Source: hawaii_case_tables.md -- all H/mva values read from hawaii.json
# -----------------------------------------------------------------------

# --- v5: 16K samples, normal std=12%, bus-diverse (2_1, 23_1, 34_1, 35_1) ---
# _H_STD_PCT = 0.12
# N_SAMPLES = 16000
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "2_1",  "param": "H", "dist": "normal", "mean": 3.69, "std": 3.69 * _H_STD_PCT},
#     {"class": "Genrou", "id": "23_1", "param": "H", "dist": "normal", "mean": 6.15, "std": 6.15 * _H_STD_PCT},
#     {"class": "Genrou", "id": "34_1", "param": "H", "dist": "normal", "mean": 4.35, "std": 4.35 * _H_STD_PCT},
#     {"class": "Genrou", "id": "35_1", "param": "H", "dist": "normal", "mean": 5.22, "std": 5.22 * _H_STD_PCT},
# ]

# # --- v6: smallest H (inertia constant) -- one per bus, distinct dynamics ---
# # Buses: 37 (H=2.48), 26 (H=3.0, small mva), 27 (H=3.0, north shore), 33 (H=3.0, west)
# _H_PCT = 0.20
# N_SAMPLES = 4000
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "37_3", "param": "H", "dist": "uniform", "nominal": 2.48, "pct": _H_PCT},  # KAHE138,     hop 2, mva=95.9,  H*mva=237.8, p0=55.9 MW
#     {"class": "Genrou", "id": "26_2", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # EWA BEACH69, hop 2, mva=11.2,  H*mva=33.6,  p0=1.1 MW
#     {"class": "Genrou", "id": "27_1", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # KAHUKU69,    far,   mva=33.0,  H*mva=99.0,  p0=9.9 MW
#     {"class": "Genrou", "id": "33_1", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # WAIANAE69,   hop 3+,mva=30.4,  H*mva=91.2,  p0=8.4 MW
# ]

# # --- v7: farthest from fault (electrical periphery, north shore + west) ---
# # Buses: 27 (north), 28 (north), 33 (west), 34 (central, hop 3+)
# _H_PCT = 0.20
# N_SAMPLES = 4000
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "27_1", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # KAHUKU69,    far north, mva=33.0,  H*mva=99.0,  p0=9.9 MW
#     {"class": "Genrou", "id": "28_1", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # HALEIWA69,   far north, mva=53.9,  H*mva=161.7, p0=25.0 MW
#     {"class": "Genrou", "id": "33_1", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # WAIANAE69,   west,      mva=30.4,  H*mva=91.2,  p0=8.4 MW
#     {"class": "Genrou", "id": "34_1", "param": "H", "dist": "uniform", "nominal": 4.35, "pct": _H_PCT},  # SCHOFIELD69, central,   mva=9.2,   H*mva=40.0,  p0=0.7 MW
# ]

# # --- v8: smallest mva (machine rating) -- one per bus ---
# # Buses: 2 (mva=2.8), 36 (mva=3.5), 34 (mva=9.2), 26 (mva=11.2)
# # Also the smallest-dispatch machines; confounded but tests "does tiny machine size matter?"
# _H_PCT = 0.20
# N_SAMPLES = 4000
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "2_1",  "param": "H", "dist": "uniform", "nominal": 3.69, "pct": _H_PCT},  # ALOHA69,     hop 1,  mva=2.8,  H*mva=10.3,  p0=0.07 MW
#     {"class": "Genrou", "id": "36_1", "param": "H", "dist": "uniform", "nominal": 4.42, "pct": _H_PCT},  # COGEN69,     hop 3+, mva=3.5,  H*mva=15.5,  p0=0.07 MW
#     {"class": "Genrou", "id": "34_1", "param": "H", "dist": "uniform", "nominal": 4.35, "pct": _H_PCT},  # SCHOFIELD69, hop 3+, mva=9.2,  H*mva=40.0,  p0=0.66 MW
#     {"class": "Genrou", "id": "26_2", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # EWA BEACH69, hop 2,  mva=11.2, H*mva=33.6,  p0=1.14 MW
# ]

# # --- v9: largest H*mva (positive control -- expect strongest H sensitivity in signals) ---
# # Buses: 35 (hop 1), 23 (hop 2), 37 (hop 2), 28 (far north)
# # If H variation is undetectable even here, the issue is elsewhere (observability, metric)
# _H_PCT = 0.20
# N_SAMPLES = 4000
# PARAM_SPECS = [
#     {"class": "Genrou", "id": "35_8", "param": "H", "dist": "uniform", "nominal": 5.22, "pct": _H_PCT},  # KALAELOA138, hop 1,     mva=124.3, H*mva=648.9, p0=69.9 MW
#     {"class": "Genrou", "id": "23_1", "param": "H", "dist": "uniform", "nominal": 6.15, "pct": _H_PCT},  # WAIPAHU69,   hop 2,     mva=85.6,  H*mva=526.4, p0=59.3 MW
#     {"class": "Genrou", "id": "37_3", "param": "H", "dist": "uniform", "nominal": 2.48, "pct": _H_PCT},  # KAHE138,     hop 2,     mva=95.9,  H*mva=237.8, p0=55.9 MW
#     {"class": "Genrou", "id": "28_2", "param": "H", "dist": "uniform", "nominal": 3.00, "pct": _H_PCT},  # HALEIWA69,   far north, mva=75.9,  H*mva=227.7, p0=52.4 MW
# ]

# --- v10: small p0 (lightly dispatched), moderate H*mva -- partial p0 isolation test ---
# Run AFTER reviewing v9 results. If v9 shows no H sensitivity at all, skip v10.
#
# Key same-bus pairs vs v9 (same bus + same H where possible, only mva/p0 differ):
#   35_4 (p0=4.4 MW, H*mva=114.8) vs 35_8 in v9 (p0=69.9 MW, H*mva=648.9) -- same bus, same H=5.22
#   23_9 (p0=2.0 MW, H*mva=48.6)  vs 23_1 in v9 (p0=59.3 MW, H*mva=526.4) -- same bus 23
#
# Hypothesis: even with moderate H*mva, lightly dispatched machines (p0~0) have negligible
# mechanical contribution to post-fault swing; H variation will be undetectable.
# Caveat: p0 and H*mva are still correlated in this snapshot -- partial test only.
# A true p0 isolation requires different operating-point snapshots (future work).
_H_PCT = 0.20
N_SAMPLES = 4000
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "35_4",
        "param": "H",
        "dist": "uniform",
        "nominal": 5.22,
        "pct": _H_PCT,
    },  # KALAELOA138, hop 1,     mva=22.0,  H*mva=114.8, p0=4.4 MW  (cf v9: 35_8, p0=69.9 MW)
    {
        "class": "Genrou",
        "id": "23_9",
        "param": "H",
        "dist": "uniform",
        "nominal": 3.00,
        "pct": _H_PCT,
    },  # WAIPAHU69,   hop 2,     mva=16.2,  H*mva=48.6,  p0=2.0 MW  (cf v9: 23_1, p0=59.3 MW)
    {
        "class": "Genrou",
        "id": "26_1",
        "param": "H",
        "dist": "uniform",
        "nominal": 3.00,
        "pct": _H_PCT,
    },  # EWA BEACH69, hop 2,     mva=22.0,  H*mva=66.0,  p0=4.4 MW
    {
        "class": "Genrou",
        "id": "27_2",
        "param": "H",
        "dist": "uniform",
        "nominal": 3.00,
        "pct": _H_PCT,
    },  # KAHUKU69,    far north, mva=30.4,  H*mva=91.2,  p0=8.4 MW
]

# -----------------------------------------------------------------------
MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR: {BASE_CASE_DIR}")
print(f"CASE_JSON:     {CASE_JSON}  (exists: {os.path.exists(CASE_JSON)})")
print(f"SOLVER_JSON:   {SOLVER_JSON}  (exists: {os.path.exists(SOLVER_JSON)})")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")
_dist = PARAM_SPECS[0]["dist"]
if _dist == "uniform":
    print(f"H dist: uniform +/-{PARAM_SPECS[0]['pct']*100:.0f}% of nominal")
    for s in PARAM_SPECS:
        print(
            f"  {s['id']:8s}: nominal={s['nominal']:.4f}  [{s['nominal']*(1-s['pct']):.4f}, {s['nominal']*(1+s['pct']):.4f}]"
        )
else:
    print(
        f"H dist: normal, std={PARAM_SPECS[0]['std']/PARAM_SPECS[0]['mean']*100:.0f}% of nominal"
    )
    for s in PARAM_SPECS:
        print(f"  {s['id']:8s}: mean={s['mean']:.4f}, std={s['std']:.4f}")

print(f"\n--- {os.path.basename(SOLVER_JSON)} ---")
print(open(SOLVER_JSON).read())

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

BASE_CASE_DIR: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/
CASE_JSON:     /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/hawaii.json  (exists: True)
SOLVER_JSON:   /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/hawaii.solver.json  (exists: True)
UQ_RUN_ROOT:   /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v10
SERIALIZE_MODE:per_run
N_SAMPLES:     4000
H dist: uniform +/-20% of nominal
  35_4    : nominal=5.2200  [4.1760, 6.2640]
  23_9    : nominal=3.0000  [2.4000, 3.6000]
  26_1    : nominal=3.0000  [2.4000, 3.6000]
  27_2    : nominal=3.0000  [2.4000, 3.6000]

--- hawaii.solver.json ---
{
    "system_model_file": "hawaii.json",
    "dt": 0.00416666666666,
    "tmax": 10.0,
    "events": [
        { "time": 1.0, "type": "fault_on", "element_id": 0 },
        { "time": 1.1, "type": "fault_off", "element_id": 0 }
    ]
}



### hawaii setup from CSV

Use this cell **instead of** the standard hawaii setup when samples are provided externally
(e.g. a Morris sensitivity design from a collaborator).

It replaces both the **write `meta.yml`** and **generate samples** cells. After running it,
go directly to **create run dirs**.

**Expected CSV format:**
```
,Genrou_2_1_H,Genrou_2_2_H,Genrou_2_3_H,...,Genrou_37_5_H
0,4.5231,3.1842,2.6543,...,2.8765
1,4.3891,3.4512,2.7231,...,2.9123
2,4.6754,3.0123,2.5891,...,2.8234
...
```
- First column: unnamed index (0, 1, 2, ...)
- Header columns: `Genrou_BUS_UNIT_H` format (validation enforces this)
- Values: numeric H parameter samples

Workflow:
1. Run this cell (sets `UQ_RUN_ROOT`, `samples_df`, `PARAM_SPECS`, writes `meta.yml`)
2. **Skip** the write-meta and generate-samples cells below
3. Continue from **create run dirs** as normal

In [54]:
# === UQ config (Hawaii) — samples from external CSV ===
import copy
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import (
    validate_csv_columns,
    deep_merge,
    generate_samples,
    make_run_dir,
    run_sample,
    collect_and_save,
)

# --- case files (same as standard hawaii setup) ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Medium/Hawaii/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "hawaii.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "hawaii.solver.json")

# --- run root and source CSV ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv"
SOURCE_CSV = "/kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/hawaii_morris_sensitivity_analysis.csv"

SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = os.path.join(UQ_RUN_ROOT, "results")
SOLVER_OVERRIDES = None

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

# --- validate CSV column naming ---
print(f"Validating CSV columns from {SOURCE_CSV}...")
cols_raw = validate_csv_columns(SOURCE_CSV, expected_class="Genrou", expected_param="H")
print(f"✓ Column validation passed: {len(cols_raw)} Genrou_*_*_H columns detected")

# --- load samples ---
# CSV has an unnamed index column + Genrou_ID_H columns (Genrou_23_1_H, etc.)
# same format as generate_samples() output -- make_run_dir reads these directly
samples_df = pd.read_csv(SOURCE_CSV, index_col=0)
N_SAMPLES = len(samples_df)
print(f"Loaded {N_SAMPLES} samples, {len(samples_df.columns)} columns")
print(f"Columns: {list(samples_df.columns)}")

# --- build PARAM_SPECS from CSV column names ---
# Column format: Genrou_BUS_UNIT_H (e.g. Genrou_23_1_H -> class=Genrou, id=23_1, param=H)
PARAM_SPECS = []
for col in samples_df.columns:
    parts = col.split("_")  # ["Genrou", "23", "1", "H"]
    if len(parts) >= 3 and parts[0] == "Genrou" and parts[-1] == "H":
        gen_id = "_".join(parts[1:-1])  # "23_1"
        PARAM_SPECS.append(
            {
                "class": "Genrou",
                "id": gen_id,
                "param": "H",
                "dist": "from_csv",
                "nominal": round(float(samples_df[col].median()), 6),
            }
        )
print(f"\nPARAM_SPECS: {len(PARAM_SPECS)} generators")
for s in PARAM_SPECS:
    print(f"  {s['id']:8s}: median H = {s['nominal']:.4f}")

# --- write meta.yml and samples.csv so collect_parallel works ---
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
_meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "sampling": {
        "n_samples": N_SAMPLES,
        "method": "from_csv",
        "dist_type": "from_csv",
        "source_csv": SOURCE_CSV,
        "params": [
            {"id": s["id"], "dist": "from_csv", "nominal": s["nominal"]}
            for s in PARAM_SPECS
        ],
    },
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "monitors_by_class": MONITORS_BY_CLASS,
}
_meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(_meta_path, "w") as f:
    yaml.dump(_meta, f, default_flow_style=False, sort_keys=False)

# collect_parallel requires samples.csv in the run root
_samples_csv_path = os.path.join(UQ_RUN_ROOT, "samples.csv")
samples_df.to_csv(_samples_csv_path)

print(f"\nWrote {_meta_path}")
print(f"Wrote {_samples_csv_path}")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")

# Show solver.json: before and after (if overrides applied)
with open(SOLVER_JSON) as f:
    solver_base = json.load(f)

print(f"\n--- {os.path.basename(SOLVER_JSON)} (base) ---")
print(json.dumps(solver_base, indent=2))

if SOLVER_OVERRIDES is not None:
    solver_modified = deep_merge(solver_base, SOLVER_OVERRIDES)
    print(f"\n--- {os.path.basename(SOLVER_JSON)} (with overrides applied) ---")
    print(json.dumps(solver_modified, indent=2))

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

Validating CSV columns from /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/hawaii_morris_sensitivity_analysis.csv...
✓ Column validation passed: 39 Genrou_*_*_H columns detected
Loaded 1000 samples, 39 columns
Columns: ['Genrou_2_1_H', 'Genrou_2_2_H', 'Genrou_2_3_H', 'Genrou_2_4_H', 'Genrou_23_1_H', 'Genrou_23_10_H', 'Genrou_23_2_H', 'Genrou_23_3_H', 'Genrou_23_4_H', 'Genrou_23_5_H', 'Genrou_23_6_H', 'Genrou_23_7_H', 'Genrou_23_8_H', 'Genrou_23_9_H', 'Genrou_26_1_H', 'Genrou_26_2_H', 'Genrou_27_1_H', 'Genrou_27_2_H', 'Genrou_28_1_H', 'Genrou_28_2_H', 'Genrou_33_1_H', 'Genrou_34_1_H', 'Genrou_34_2_H', 'Genrou_34_3_H', 'Genrou_34_4_H', 'Genrou_34_5_H', 'Genrou_34_6_H', 'Genrou_35_1_H', 'Genrou_35_2_H', 'Genrou_35_4_H', 'Genrou_35_5_H', 'Genrou_35_7_H', 'Genrou_35_8_H', 'Genrou_36_1_H', 'Genrou_36_2_H', 'Genrou_36_3_H', 'Genrou_36_4_H', 'Genrou_37_3_H', 'Genrou_37_5_H']

PARAM_SPECS: 39 generators
  2_1     : median H = 3.9360
  2_2     : median H = 3.4440
  2_3     : media

### illinois setup

In [ ]:
# === UQ config (Illinois) ===
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import generate_samples, make_run_dir, run_sample, collect_and_save

# --- case files ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Large/Illinois/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "illinois.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "illinois.solver.json")

# --- output ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2"
# "per_run" (default): one run_NNN.parquet per run under runs/ -- use for N>=100
# "stacked": all runs in one results.parquet -- only for small N (<500)
SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = (
    os.path.join(UQ_RUN_ROOT, "results.parquet")
    if SERIALIZE_MODE == "stacked"
    else os.path.join(UQ_RUN_ROOT, "runs")
)

# --- sampling ---
N_SAMPLES = 4000
SEED = 42
SAMPLE_METHOD = "lhs"  # "lhs" or "random"

# --- solver overrides ---
# base illinois.solver.json: tmax=20.0 s, fault at t=10.0/10.1 s
# override to match Hawaii cadence: fault early, shorter sim
# Note: use full event replacement (no "index" needed); much cleaner than selective patching
SOLVER_OVERRIDES = {
    "tmax": 10.0,
    "events": [
        {"time": 1.0, "type": "fault_on", "element_id": 0},
        {"time": 1.1, "type": "fault_off", "element_id": 0},
    ],
}

# -----------------------------------------------------------------------
# H parameter specs: 4 generators at increasing hop distance from fault (bus 2)
# hop 4: 126_1 BARTONVILLE 3  (coal, 130 MW)  H=7.29971
# hop 5: 135_1 PEKIN 1 2      (coal, 446 MW)  H=3.23290
# hop 7: 115_1 NORMAL 2 3     (wind, 133 MW)  H=2.72678
# hop 9: 189_1 CLINTON 1 2    (nuclear, 569 MW) H=3.40165
# -----------------------------------------------------------------------

# --- epistemic -- Gaussian std=12% of nominal ---
_H_STD_PCT = 0.12
PARAM_SPECS = [
    {
        "class": "Genrou",
        "id": "126_1",
        "param": "H",
        "dist": "normal",
        "mean": 7.29971075,
        "std": 7.29971075 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "135_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.23289871,
        "std": 3.23289871 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "115_1",
        "param": "H",
        "dist": "normal",
        "mean": 2.72677827,
        "std": 2.72677827 * _H_STD_PCT,
    },
    {
        "class": "Genrou",
        "id": "189_1",
        "param": "H",
        "dist": "normal",
        "mean": 3.40165448,
        "std": 3.40165448 * _H_STD_PCT,
    },
]

# -----------------------------------------------------------------------
# Monitored variables: make_run_dir patches the mon[] array in each
# bus/device entry of the copied case JSON with these values, overwriting
# the base case defaults (Vr, Vi).
#
# bus:    Vm = voltage magnitude (pu), Va = voltage angle (rad)
#         alternatives: Vr = real part, Vi = imaginary part
# genrou: delta = rotor angle (rad), omega = speed deviation (pu, ~0 steady-state)
# -----------------------------------------------------------------------
MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

print(f"BASE_CASE_DIR: {BASE_CASE_DIR}")
print(f"CASE_JSON:     {CASE_JSON}  (exists: {os.path.exists(CASE_JSON)})")
print(f"SOLVER_JSON:   {SOLVER_JSON}  (exists: {os.path.exists(SOLVER_JSON)})")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")
print(f"SOLVER_OVERRIDES: {SOLVER_OVERRIDES}")
print(f"H dist: normal, std={_H_STD_PCT*100:.0f}% of nominal")
for s in PARAM_SPECS:
    print(f"  {s['id']}: mean={s['mean']:.5f}, std={s['std']:.5f}")

print(f"\n--- {os.path.basename(SOLVER_JSON)} (base, before overrides) ---")
print(open(SOLVER_JSON).read())

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

BASE_CASE_DIR: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/
CASE_JSON:     /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/illinois.json  (exists: True)
SOLVER_JSON:   /home/isatkaus/gridkit/build/examples/PhasorDynamics/Large/Illinois/illinois.solver.json  (exists: True)
UQ_RUN_ROOT:   /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-v2
SERIALIZE_MODE:per_run
N_SAMPLES:     4000
SOLVER_OVERRIDES: {'tmax': 10.0, 'events': [{'index': 0, 'time': 1.0}, {'index': 1, 'time': 1.1}]}
H dist: normal, std=12% of nominal
  126_1: mean=7.29971, std=0.87597
  135_1: mean=3.23290, std=0.38795
  115_1: mean=2.72678, std=0.32721
  189_1: mean=3.40165, std=0.40820

--- illinois.solver.json (base, before overrides) ---
{
    "system_model_file": "illinois.json",
    "dt": 0.00416666666666,
    "tmax": 20.0,
    "events": [
        { "time": 10.0, "type": "fault_on", "element_id": 0 },
        { "time": 10.1, "type": "fault_off", "element_id": 0 }
    ]

### illinois setup from CSV

Use this cell **instead of** the standard illinois setup when samples are provided externally
(e.g. a Morris sensitivity design from a collaborator).

It replaces both the **write `meta.yml`** and **generate samples** cells. After running it,
go directly to **create run dirs**.

**Expected CSV format:**
```
,Genrou_49_1_H,Genrou_50_1_H,Genrou_51_1_H,...,Genrou_197_1_H
0,7.4523,3.2154,2.8912,...,3.5678
1,7.2891,3.4567,2.6543,...,3.4123
2,7.5234,3.1890,2.9234,...,3.6145
...
```
- First column: unnamed index (0, 1, 2, ...)
- Header columns: `Genrou_BUS_UNIT_H` format (validation enforces this)
- Values: numeric H parameter samples

Workflow:
1. Run this cell (sets `UQ_RUN_ROOT`, `samples_df`, `PARAM_SPECS`, writes `meta.yml`)
2. **Skip** the write-meta and generate-samples cells below
3. Continue from **create run dirs** as normal

In [8]:
# === UQ config (Illinois) — samples from external CSV ===
import copy
import importlib
import gridkit_utils

importlib.reload(gridkit_utils)
from gridkit_utils import (
    validate_csv_columns,
    deep_merge,
    generate_samples,
    make_run_dir,
    run_sample,
    collect_and_save,
)

# --- case files (same as standard illinois setup) ---
BASE_CASE_DIR = os.path.join(build_dir, "examples/PhasorDynamics/Large/Illinois/")
CASE_JSON = os.path.join(BASE_CASE_DIR, "illinois.json")
SOLVER_JSON = os.path.join(BASE_CASE_DIR, "illinois.solver.json")

# --- run root and source CSV ---
UQ_RUN_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv"
SOURCE_CSV = "/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/illinois_morris_design_fixed.csv"

SERIALIZE_MODE = "per_run"
UQ_OUT_PATH = os.path.join(UQ_RUN_ROOT, "results")
SOLVER_OVERRIDES = {
    "tmax": 10.0,
    "events": [
        {"time": 1.0, "type": "fault_on", "element_id": 0},
        {"time": 1.1, "type": "fault_off", "element_id": 0},
    ],
}

MONITORS_BY_CLASS = {
    "bus": ["Vm", "Va"],
    "genrou": ["delta", "omega"],
}

# --- validate CSV column naming ---
print(f"Validating CSV columns from {SOURCE_CSV}...")
cols_raw = validate_csv_columns(SOURCE_CSV, expected_class="Genrou", expected_param="H")
print(f"✓ Column validation passed: {len(cols_raw)} Genrou_*_*_H columns detected")

# --- load samples ---
# CSV has an unnamed index column + Genrou_ID_H columns (Genrou_126_1_H, etc.)
# same format as generate_samples() output -- make_run_dir reads these directly
samples_df = pd.read_csv(SOURCE_CSV, index_col=0)
N_SAMPLES = len(samples_df)
print(f"Loaded {N_SAMPLES} samples, {len(samples_df.columns)} columns")
print(f"Columns: {list(samples_df.columns)}")

# --- build PARAM_SPECS from CSV column names ---
# Column format: Genrou_BUS_UNIT_H (e.g. Genrou_126_1_H -> class=Genrou, id=126_1, param=H)
PARAM_SPECS = []
for col in samples_df.columns:
    parts = col.split("_")  # ["Genrou", "126", "1", "H"]
    if len(parts) >= 3 and parts[0] == "Genrou" and parts[-1] == "H":
        gen_id = "_".join(parts[1:-1])  # "126_1"
        PARAM_SPECS.append(
            {
                "class": "Genrou",
                "id": gen_id,
                "param": "H",
                "dist": "from_csv",
                "nominal": round(float(samples_df[col].median()), 6),
            }
        )
print(f"\nPARAM_SPECS: {len(PARAM_SPECS)} generators")
for s in PARAM_SPECS:
    print(f"  {s['id']:8s}: median H = {s['nominal']:.4f}")

# --- write meta.yml and samples.csv so collect_parallel works ---
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
_meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "sampling": {
        "n_samples": N_SAMPLES,
        "method": "from_csv",
        "dist_type": "from_csv",
        "source_csv": SOURCE_CSV,
        "params": [
            {"id": s["id"], "dist": "from_csv", "nominal": s["nominal"]}
            for s in PARAM_SPECS
        ],
    },
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "monitors_by_class": MONITORS_BY_CLASS,
}
_meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(_meta_path, "w") as f:
    yaml.dump(_meta, f, default_flow_style=False, sort_keys=False)

# collect_parallel requires samples.csv in the run root
_samples_csv_path = os.path.join(UQ_RUN_ROOT, "samples.csv")
samples_df.to_csv(_samples_csv_path)

print(f"\nWrote {_meta_path}")
print(f"Wrote {_samples_csv_path}")
print(f"UQ_RUN_ROOT:   {UQ_RUN_ROOT}")
print(f"SERIALIZE_MODE:{SERIALIZE_MODE}")
print(f"N_SAMPLES:     {N_SAMPLES}")

# Show solver.json: before and after (if overrides applied)
with open(SOLVER_JSON) as f:
    solver_base = json.load(f)

print(f"\n--- {os.path.basename(SOLVER_JSON)} (base) ---")
print(json.dumps(solver_base, indent=2))

if SOLVER_OVERRIDES is not None:
    solver_modified = deep_merge(solver_base, SOLVER_OVERRIDES)
    print(f"\n--- {os.path.basename(SOLVER_JSON)} (with overrides applied) ---")
    print(json.dumps(solver_modified, indent=2))

<module 'gridkit_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/gridkit_utils.py'>

Validating CSV columns from /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/illinois_morris_design_fixed.csv...
✓ Column validation passed: 40 Genrou_*_*_H columns detected
Loaded 1025 samples, 40 columns
Columns: ['Genrou_49_1_H', 'Genrou_50_1_H', 'Genrou_51_1_H', 'Genrou_52_1_H', 'Genrou_53_1_H', 'Genrou_65_1_H', 'Genrou_67_1_H', 'Genrou_68_1_H', 'Genrou_69_1_H', 'Genrou_70_1_H', 'Genrou_71_1_H', 'Genrou_72_1_H', 'Genrou_73_1_H', 'Genrou_76_1_H', 'Genrou_77_1_H', 'Genrou_90_1_H', 'Genrou_91_1_H', 'Genrou_94_1_H', 'Genrou_104_1_H', 'Genrou_105_1_H', 'Genrou_114_1_H', 'Genrou_115_1_H', 'Genrou_125_1_H', 'Genrou_126_1_H', 'Genrou_127_1_H', 'Genrou_135_1_H', 'Genrou_136_1_H', 'Genrou_147_1_H', 'Genrou_151_1_H', 'Genrou_152_1_H', 'Genrou_153_1_H', 'Genrou_154_1_H', 'Genrou_155_1_H', 'Genrou_161_1_H', 'Genrou_167_1_H', 'Genrou_170_1_H', 'Genrou_182_1_H', 'Genrou_183_1_H', 'Genrou_189_1_H', 'Genrou_197_1_H']

PARAM_SPECS: 40 generators
  49_1    : median H = 4.7945
  50_1   

## experiment steps

Run these cells in order after selecting a case config above.

1. Write `meta.yml`
2. Generate `samples.csv`
   > **Skip steps 1 and 2** if you used the **hawaii setup from CSV** or **illinois setup from CSV** cell — `meta.yml` and `samples_df` are already set.
3. Create run dirs (patch `case.json` + `solver.json` for each sample)
4. **Run simulations** — choose ONE option:
   - **Serial** (next section): run all samples in a loop on this node; fine for small N or quick tests
   - **SLURM** (section after): write + submit bash array jobs; use for large N or large cases
5. Once all simulations complete, continue with **collect results** (shared regardless of run method)

In [20]:
# === Write experiment metadata to meta.yml ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)

_dist = PARAM_SPECS[0]["dist"]
_sampling_summary = []
for s in PARAM_SPECS:
    if s["dist"] == "uniform":
        lo = s["nominal"] * (1 - s["pct"])
        hi = s["nominal"] * (1 + s["pct"])
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "uniform",
                "nominal": s["nominal"],
                "pct": s["pct"],
                "lo": round(lo, 6),
                "hi": round(hi, 6),
            }
        )
    else:
        _sampling_summary.append(
            {
                "id": s["id"],
                "dist": "normal",
                "mean": s["mean"],
                "std": round(s["std"], 6),
                "std_pct": round(s["std"] / s["mean"], 4),
            }
        )

meta = {
    "case": os.path.basename(BASE_CASE_DIR.rstrip("/")),
    "base_case_dir": BASE_CASE_DIR,
    "run_root": UQ_RUN_ROOT,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "sampling": {
        "n_samples": N_SAMPLES,
        "seed": SEED,
        "method": SAMPLE_METHOD,
        "dist_type": _dist,
        "params": _sampling_summary,
    },
    "serialize_mode": SERIALIZE_MODE,
    "solver_overrides": SOLVER_OVERRIDES,
    "monitors_by_class": MONITORS_BY_CLASS,
}
meta_path = os.path.join(UQ_RUN_ROOT, "meta.yml")
with open(meta_path, "w") as f:
    yaml.dump(meta, f, default_flow_style=False, sort_keys=False)
print(f"Wrote {meta_path}")
print(open(meta_path).read())

Wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v10/meta.yml
case: Hawaii
base_case_dir: /home/isatkaus/gridkit/build/examples/PhasorDynamics/Medium/Hawaii/
run_root: /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v10
created: '2026-08-11T09:20:16'
sampling:
  n_samples: 4000
  seed: 42
  method: lhs
  dist_type: uniform
  params:
  - id: '35_4'
    dist: uniform
    nominal: 5.22
    pct: 0.2
    lo: 4.176
    hi: 6.264
  - id: '23_9'
    dist: uniform
    nominal: 3.0
    pct: 0.2
    lo: 2.4
    hi: 3.6
  - id: '26_1'
    dist: uniform
    nominal: 3.0
    pct: 0.2
    lo: 2.4
    hi: 3.6
  - id: '27_2'
    dist: uniform
    nominal: 3.0
    pct: 0.2
    lo: 2.4
    hi: 3.6
serialize_mode: per_run
solver_overrides: null
monitors_by_class:
  bus:
  - Vm
  - Va
  genrou:
  - delta
  - omega



In [21]:
# === Generate samples ===
os.makedirs(UQ_RUN_ROOT, exist_ok=True)
samples_df = generate_samples(PARAM_SPECS, N=N_SAMPLES, seed=SEED, method=SAMPLE_METHOD)
samples_df.to_csv(os.path.join(UQ_RUN_ROOT, "samples.csv"))
print(f"samples_df shape: {samples_df.shape}")
samples_df

samples_df shape: (4000, 4)


,Genrou_35_4_H,Genrou_23_9_H,Genrou_26_1_H,Genrou_27_2_H
0,4.224142,3.501168,2.916042,3.106291
1,5.180279,3.324607,3.557172,2.778664
2,5.586899,2.719965,2.574189,2.718022
3,4.727418,3.330953,2.933567,2.740432
4,4.304123,2.513681,2.957752,2.818011
...,...,...,...,...
3995,4.861891,2.518738,3.327688,2.638175
3996,5.730901,3.398054,3.057383,2.631426
3997,6.160088,2.899662,3.570537,3.301510
3998,5.824420,2.990821,3.269555,3.331027


In [6]:
# === Create run dirs + patch case.json ===
# (start here, if "setup from csv")
# Runs are organized under sample_runs/ with dynamic padding based on N_SAMPLES
_n_digits = len(str(N_SAMPLES - 1))
_fmt = f"run_{{i:0{_n_digits}d}}"

for i, row in samples_df.iterrows():
    d = make_run_dir(
        BASE_CASE_DIR,
        UQ_RUN_ROOT,
        i,
        row,
        PARAM_SPECS,
        MONITORS_BY_CLASS,
        solver_overrides=SOLVER_OVERRIDES,
        n_samples=N_SAMPLES,
    )
    if i % 100 == 0:
        print(f"  {_fmt.format(i=i)}: {d}")
print(
    f"Created {N_SAMPLES} run dirs (with {_n_digits}-digit padding) in {UQ_RUN_ROOT}/sample_runs/"
)

  run_0000: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0000
  run_0100: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0100
  run_0200: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0200
  run_0300: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0300
  run_0400: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0400
  run_0500: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0500
  run_0600: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0600
  run_0700: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0700
  run_0800: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0800
  run_0900: /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/sample_runs/run_0900
  run_1000: /kfs2/projects/sci

### UQ Experiment Directory Structure

**OLD (flat, cluttered):**
```
UQ_RUN_ROOT/
  meta.yml
  samples.csv
  run_000/
  run_001/
  run_002/
  ...
  run_999/  ← 1000+ directories makes ls slow and hard to navigate!
```

**NEW (organized with subdirectories):**
```
UQ_RUN_ROOT/
  meta.yml          ← experiment configuration
  samples.csv       ← input parameters for each run
  sample_runs/      ← run directories (hidden from simple ls)
    run_0000/, run_0001/, ..., run_1024/
      case.json   ← for each run one H is diferent
      solver.json
      mon.csv
  results/          ← collected results (Parquet format)
    run_0000.parquet, run_0001.parquet, ..., run_1024.parquet
```

**Key points:**
- `ls UQ_RUN_ROOT` shows ~5 items, not 1000+ directories
- Padding width auto-scales: 100 samples → 2-digit, 1000 → 3-digit, 10000+ → 4-digit
- Consistent file naming for easy sorting and correlation

### option A: serial run

Run all samples in a loop on this node.
Good for small N (up to ~200) or quick tests.
**Skip this cell** and go to option B below for large N or large grid cases.

In [27]:
# # === Run all samples ===
# failed = []
# for i, row in samples_df.iterrows():
#     run_dir = os.path.join(UQ_RUN_ROOT, f"run_{i:03d}")
#     result = run_sample(run_dir, runner)
#     status = "OK" if result.returncode == 0 else "FAILED"
#     if result.returncode != 0:
#         failed.append(i)
#         print(f"  run_{i:03d}: FAILED  {result.stderr[:200]}")
#     elif i % 100 == 0:
#         print(f"  run_{i:03d}: OK")

# print(f"\nDone. {N_SAMPLES - len(failed)}/{N_SAMPLES} succeeded.")
# if failed:
#     print(f"Failed runs: {failed}")

### option B: SLURM parallel run

Run samples as bash array jobs across multiple cluster nodes.
Use for large N (1K+) or large grid cases where serial is too slow.
**Skip this section** if you ran option A above.

Steps:
1. Set SLURM config (nodes, partition, account, walltime, workers per node)
2. Write sbatch scripts — one bash script per node, each loops over its sample slice using `xargs -P`
3. Submit all scripts with `sbatch`
4. Re-run the status cell until all jobs complete, then continue with **collect results** below

#### SLURM config

In [9]:
# === SLURM config ===
SLURM_ACCOUNT = "scidac"  # --account  (your project handle)
SLURM_PARTITION = "short"  # --partition
WALLTIME = "01:00:00"  # --time
N_NODES = 4  # number of sbatch scripts (one per node)
WORKERS_PER_NODE = 4  # parallel DynamicSimulation per node via xargs -P
# keep <=104 (Kestrel CPUs/node); use 1 for large grids
SLURM_QOS = None  # None = default priority
# "high"    = 2x AU cost, faster queue
# "standby" = free, only runs on idle nodes

# Modules to load inside each sbatch script.
# Add whatever DynamicSimulation needs at runtime (e.g. MKL, compiler libs).
# Leave empty if the binary is self-contained or the environment is set via conda.
SLURM_MODULES = [
    # "intel-oneapi-mkl/2024.0",
    # "gcc/13.1.0",
]

# --- collect job (option B: SLURM-side collection) ---
# If True, a separate collect.sh job is written and submitted with --dependency=afterok
# on all sim jobs. It runs on one additional node after all sims complete.
# If False, collect manually using the notebook collect cell below.
COLLECT_IN_SLURM = True
COLLECT_WORKERS = 64  # threads for SLURM collect; more aggressive than notebook
COLLECT_WALLTIME = "02:00:00"
# conda env path used by the collect job to import pandas + pyarrow
CONDA_ENV = "/home/isatkaus/flash-w-dom/conda-envs/h-py312-basic"

# Derived: slice indices per node
_chunk_size = (N_SAMPLES + N_NODES - 1) // N_NODES
_slices = [
    (i * _chunk_size, min((i + 1) * _chunk_size - 1, N_SAMPLES - 1))
    for i in range(N_NODES)
]
print(f"N_SAMPLES={N_SAMPLES}, N_NODES={N_NODES}, WORKERS_PER_NODE={WORKERS_PER_NODE}")
print(f"QOS: {SLURM_QOS or '(default)'}")
print(f"COLLECT_IN_SLURM: {COLLECT_IN_SLURM}")
print("Index slices per node:")
for node_i, (lo, hi) in enumerate(_slices):
    print(f"  node {node_i:02d}: runs {lo:04d} - {hi:04d}  ({hi - lo + 1} samples)")
if COLLECT_IN_SLURM:
    print(f"  + collect node: {COLLECT_WORKERS} threads, walltime={COLLECT_WALLTIME}")

N_SAMPLES=1025, N_NODES=4, WORKERS_PER_NODE=4
QOS: (default)
COLLECT_IN_SLURM: True
Index slices per node:
  node 00: runs 0000 - 0256  (257 samples)
  node 01: runs 0257 - 0513  (257 samples)
  node 02: runs 0514 - 0770  (257 samples)
  node 03: runs 0771 - 1024  (254 samples)
  + collect node: 64 threads, walltime=02:00:00


In [9]:
# === Write sbatch scripts ===
# Generate bash scripts for simulation jobs (one per node) + optional collect job.
# Each script runs samples in parallel using xargs -P WORKERS_PER_NODE.

_n_digits = len(str(N_SAMPLES - 1))  # Dynamic padding based on N_SAMPLES
_slurm_dir = os.path.join(UQ_RUN_ROOT, "slurm")
os.makedirs(_slurm_dir, exist_ok=True)

_script_paths = []
for node_i, (lo, hi) in enumerate(_slices):
    _indices = " ".join(f"{i:0{_n_digits}d}" for i in range(lo, hi + 1))
    script = f"""#!/bin/bash
#SBATCH --job-name="sim_{node_i:02d}"
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={WORKERS_PER_NODE}
#SBATCH --time={WALLTIME}
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
#SBATCH --output={_slurm_dir}/chunk_{node_i:02d}.log
{"" if not SLURM_QOS else f"#SBATCH --qos={SLURM_QOS}"}

set -e
export PATH={GRIDKIT_BUILD_DIR}/bin:$PATH
export PYTHONPATH={GRIDKIT_PY_UTILS}:$PYTHONPATH

# Module loads (if any)
{"" if not SLURM_MODULES else "module load " + " ".join(SLURM_MODULES)}

cd {UQ_RUN_ROOT}

INDICES=({_indices})
printf '%s\\n' "${{INDICES[@]}}" | xargs -P {WORKERS_PER_NODE} -I {{}} bash -c '
    dir="sample_runs/run_{{}}"
    solver=$(ls "$dir"/*.solver.json 2>/dev/null | head -1)
    if [[ -z "$solver" ]]; then
        echo "NO_SOLVER $dir" >> "{_slurm_dir}/chunk_{node_i:02d}_failed.txt"
        exit 0
    fi
    cd "$dir"
    {runner} "$(basename $solver)" > stdout.txt 2> stderr.txt \\
        || echo "FAILED $dir" >> "{_slurm_dir}/chunk_{node_i:02d}_failed.txt"
'
"""
    script_path = os.path.join(_slurm_dir, f"chunk_{node_i:02d}.sh")
    with open(script_path, "w") as f:
        f.write(script)
    os.chmod(script_path, 0o755)
    _script_paths.append(script_path)
    print(f"  wrote {script_path}  (runs {lo:04d}-{hi:04d})")

# Optional: write collect.sh if COLLECT_IN_SLURM
_collect_script_path = None
if COLLECT_IN_SLURM:
    collect_script = f"""#!/bin/bash
#SBATCH --job-name="collect"
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={COLLECT_WORKERS}
#SBATCH --time={COLLECT_WALLTIME}
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
#SBATCH --output={_slurm_dir}/collect.log

source /nopt/nrel/apps/software/mamba/etc/profile.d/conda.sh
conda activate {CONDA_ENV}

cd {UQ_RUN_ROOT}
python3 << 'PYEOF'
import sys
sys.path.insert(0, "{GRIDKIT_PY_UTILS}")
from gridkit_utils import collect_parallel

result, missing = collect_parallel(
    "{UQ_RUN_ROOT}",
    "{UQ_OUT_PATH}",
    n_workers={COLLECT_WORKERS},
    n_samples={N_SAMPLES},
)
print(f"Collected {{len(result)}} results")
if missing:
    print(f"Missing runs: {{missing}}")
    sys.exit(1)
PYEOF
"""
    _collect_script_path = os.path.join(_slurm_dir, "collect.sh")
    with open(_collect_script_path, "w") as f:
        f.write(collect_script)
    os.chmod(_collect_script_path, 0o755)
    print(f"    wrote {_collect_script_path}  (collect job, {COLLECT_WORKERS} threads)")

print(
    f"\nTotal nodes: {len(_script_paths)} sim"
    + (f" + 1 collect = {len(_script_paths) + 1}" if COLLECT_IN_SLURM else "")
)
print(f"\nReview sim script with:  cat {_slurm_dir}/chunk_00.sh")
if _collect_script_path:
    print(f"Review collect script:   cat {_collect_script_path}")

2398

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_00.sh  (runs 0000-0256)


2398

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_01.sh  (runs 0257-0513)


2398

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_02.sh  (runs 0514-0770)


2383

  wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_03.sh  (runs 0771-1024)


955

    wrote /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/collect.sh  (collect job, 64 threads)

Total nodes: 4 sim + 1 collect = 5

Review sim script with:  cat /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_00.sh
Review collect script:   cat /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/collect.sh


In [ ]:
# === Submit sbatch jobs ===
# Review the scripts in slurm/ before running this cell.

_job_ids = []
for script_path in _script_paths:
    cmd = ["sbatch", script_path]
    print(" ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr.strip()}")
    else:
        jid = result.stdout.strip().split()[-1]
        _job_ids.append(jid)
        print(f"  submitted job {jid}")

print(f"\nSubmitted {len(_job_ids)} sim jobs: {_job_ids}")
print(
    f"Monitor with:  sacct -j {','.join(_job_ids)} --format=JobID,State,Elapsed,ExitCode"
)

# --- submit collect job with dependency if COLLECT_IN_SLURM ---
_collect_job_id = None
if COLLECT_IN_SLURM and _collect_script_path and _job_ids:
    _dep = "afterok:" + ":".join(_job_ids)
    cmd = ["sbatch", f"--dependency={_dep}", _collect_script_path]
    print(f"\n{' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR submitting collect job: {result.stderr.strip()}")
    else:
        _collect_job_id = result.stdout.strip().split()[-1]
        print(
            f"  submitted collect job {_collect_job_id}  (runs after all sim jobs complete)"
        )
        print(f"\nAll jobs: sim={_job_ids}, collect={_collect_job_id}")
        print(f"Total: {len(_job_ids) + 1} nodes")

# --- Save job IDs to file for later recovery ---
_job_ids_file = os.path.join(_slurm_dir, "job_ids.txt")
_job_metadata = {
    "sim_job_ids": _job_ids,
    "collect_job_id": _collect_job_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "uq_run_root": UQ_RUN_ROOT,
    "n_samples": N_SAMPLES,
}
import json as json_module

with open(_job_ids_file, "w") as f:
    json_module.dump(_job_metadata, f, indent=2)
print(f"\nSaved job metadata to: {_job_ids_file}")

sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_00.sh
  submitted job 18723062
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_01.sh
  submitted job 18723063
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_02.sh
  submitted job 18723064
sbatch /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/chunk_03.sh
  submitted job 18723065

Submitted 4 sim jobs: ['18723062', '18723063', '18723064', '18723065']
Monitor with:  sacct -j 18723062,18723063,18723064,18723065 --format=JobID,State,Elapsed,ExitCode

sbatch --dependency=afterok:18723062:18723063:18723064:18723065 /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/slurm/collect.sh
  submitted collect job 18723066  (runs after all sim jobs complete)

All jobs: sim=['18723062', '18723063', '18723064', '18723065'], collect=18723066
Total: 5 nodes


In [ ]:
# === Check Job Status (Independent - can run anytime) ===
# Reads job_ids.txt saved by submit cell. Works even after kernel restart.
# Does NOT require the submit cell to have run in current session.

import json as json_module

# Try to load job IDs from file
_job_ids_file = os.path.join(UQ_RUN_ROOT, "slurm", "job_ids.txt")
if not os.path.exists(_job_ids_file):
    print(f"ERROR: {_job_ids_file} not found.")
    print("You must run the Submit cell first to save job IDs.")
else:
    with open(_job_ids_file, "r") as f:
        _job_metadata = json_module.load(f)

    _job_ids = _job_metadata.get("sim_job_ids", [])
    _collect_job_id = _job_metadata.get("collect_job_id")

    print(f"Loaded job IDs from: {_job_ids_file}")
    print(f"UQ_RUN_ROOT: {_job_metadata.get('uq_run_root')}")
    print(f"N_SAMPLES: {_job_metadata.get('n_samples')}")
    print(f"Submitted: {_job_metadata.get('timestamp')}")
    print()

    # Query sacct for sim jobs
    if _job_ids:
        cmd = [
            "sacct",
            "-j",
            ",".join(_job_ids),
            "--format=JobID,JobName,State,Elapsed,ExitCode",
        ]
        print(" ".join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)

        # Parse job states
        lines = result.stdout.strip().split("\n")[1:]  # skip header
        completed = failed = running = pending = 0
        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 3:
                state = parts[2]
                if state == "COMPLETED":
                    completed += 1
                elif state in ("FAILED", "TIMEOUT", "CANCELLED"):
                    failed += 1
                elif state == "RUNNING":
                    running += 1
                elif state in ("PENDING", "CONFIGURING"):
                    pending += 1

        print(
            f"\nStatus: {completed} COMPLETED, {running} RUNNING, {pending} PENDING, {failed} FAILED"
        )

    # Query collect job if it exists
    if _collect_job_id:
        print(f"\n--- Collect Job {_collect_job_id} ---")
        cmd = [
            "sacct",
            "-j",
            _collect_job_id,
            "--format=JobID,JobName,State,Elapsed,ExitCode",
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)

In [ ]:
# === Identify Failed Runs ===
# Parse chunk_*_failed.txt files to find which runs failed.
# Only shows failed runs if the job actually ran and created failure logs.

_n_digits = len(str(N_SAMPLES - 1))
_slurm_dir = os.path.join(UQ_RUN_ROOT, "slurm")
_failed_runs = set()
_failed_by_chunk = {}

# Search for failed.txt files
for chunk_file in sorted(glob.glob(os.path.join(_slurm_dir, "chunk_*_failed.txt"))):
    chunk_name = (
        os.path.basename(chunk_file).replace("chunk_", "").replace("_failed.txt", "")
    )
    _failed_by_chunk[chunk_name] = []

    try:
        with open(chunk_file, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    # Parse: "FAILED sample_runs/run_0042" or "NO_SOLVER sample_runs/run_0042"
                    parts = line.split()
                    if len(parts) >= 2:
                        status = parts[0]
                        run_dir = parts[1] if len(parts) > 1 else ""
                        # Extract run number from "sample_runs/run_NNNN"
                        if "run_" in run_dir:
                            run_num_str = run_dir.split("run_")[-1].rstrip("/")
                            try:
                                run_num = int(run_num_str)
                                _failed_runs.add(run_num)
                                _failed_by_chunk[chunk_name].append((run_num, status))
                            except ValueError:
                                pass
    except FileNotFoundError:
        pass

if not _failed_runs:
    print(f"✓ No failures found! All runs completed successfully.")
else:
    print(
        f"✗ Found {len(_failed_runs)} failed runs across {len(_failed_by_chunk)} chunks:"
    )
    print()
    for chunk_name in sorted(_failed_by_chunk.keys()):
        runs = _failed_by_chunk[chunk_name]
        if runs:
            print(f"  chunk_{chunk_name}: {len(runs)} failures")
            for run_num, status in sorted(runs)[:5]:  # Show first 5
                print(f"    - run_{run_num:0{_n_digits}d}: {status}")
            if len(runs) > 5:
                print(f"    ... and {len(runs) - 5} more")
    print()
    print(f"Failed run indices: {sorted(_failed_runs)}")
    print(f"\nTo resubmit, run the next cell: 'Mark Runs for Resubmit'")

In [ ]:
# === Mark Runs for Resubmit (Interactive) ===
# Select which failed runs to resubmit. Saves selection to file.

if not _failed_runs:
    print("No failed runs to resubmit. Skipping.")
else:
    print(f"Found {len(_failed_runs)} failed runs.")
    print()

    # Interactive choice
    print("Options:")
    print("  1. Resubmit ALL failed runs")
    print("  2. Resubmit NONE (cancel)")
    print("  3. Resubmit SPECIFIC runs (manual list)")
    print()

    choice = input("Enter choice (1-3, default=1): ").strip() or "1"

    _resubmit_indices = []

    if choice == "1":
        _resubmit_indices = sorted(list(_failed_runs))
        print(f"✓ Will resubmit all {len(_resubmit_indices)} failed runs")
    elif choice == "2":
        print("✓ Cancelled. No runs marked for resubmit.")
        _resubmit_indices = []
    elif choice == "3":
        indices_str = input(
            "Enter run indices (comma-separated, e.g., '5,17,42'): "
        ).strip()
        try:
            _resubmit_indices = [int(x.strip()) for x in indices_str.split(",")]
            _resubmit_indices = [i for i in _resubmit_indices if i in _failed_runs]
            print(f"✓ Will resubmit {len(_resubmit_indices)} runs: {_resubmit_indices}")
        except ValueError:
            print("ERROR: Invalid input")
            _resubmit_indices = []
    else:
        print(f"Invalid choice: {choice}")
        _resubmit_indices = []

    # Save to file
    if _resubmit_indices:
        _resubmit_file = os.path.join(_slurm_dir, "resubmit_indices.txt")
        with open(_resubmit_file, "w") as f:
            f.write("\n".join(str(i) for i in _resubmit_indices))
        print(f"\nSaved to: {_resubmit_file}")
        print(
            f"Run next cell: 'Resubmit Failed Runs' to generate and submit SLURM scripts."
        )

In [ ]:
# === Resubmit Failed Runs ===
# Generate sbatch scripts for only the failed runs and submit them.

_resubmit_file = os.path.join(_slurm_dir, "resubmit_indices.txt")
if not os.path.exists(_resubmit_file):
    print(f"No resubmit file found at {_resubmit_file}")
    print("Run 'Mark Runs for Resubmit' first.")
else:
    with open(_resubmit_file, "r") as f:
        _resubmit_indices = [int(line.strip()) for line in f if line.strip()]

    if not _resubmit_indices:
        print("No runs marked for resubmit.")
    else:
        print(f"Resubmitting {len(_resubmit_indices)} runs: {_resubmit_indices}")
        print()

        _n_digits = len(str(N_SAMPLES - 1))
        _resubmit_job_ids = []

        # Group resubmit runs into chunks (same as original N_NODES)
        _chunk_size = (len(_resubmit_indices) + N_NODES - 1) // N_NODES

        for node_i in range(N_NODES):
            lo_idx = node_i * _chunk_size
            hi_idx = min((node_i + 1) * _chunk_size, len(_resubmit_indices))

            if lo_idx >= hi_idx:
                break

            _chunk_indices = _resubmit_indices[lo_idx:hi_idx]
            _indices_str = " ".join(f"{i:0{_n_digits}d}" for i in _chunk_indices)

            # Generate resubmit script
            resubmit_script = f"""#!/bin/bash
#SBATCH --job-name="resub_{node_i:02d}"
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={WORKERS_PER_NODE}
#SBATCH --time={WALLTIME}
#SBATCH --account={SLURM_ACCOUNT}
#SBATCH --partition={SLURM_PARTITION}
#SBATCH --output={_slurm_dir}/resubmit_{node_i:02d}.log
{"" if not SLURM_QOS else f"#SBATCH --qos={SLURM_QOS}"}

set -e
export PATH={GRIDKIT_BUILD_DIR}/bin:$PATH
export PYTHONPATH={GRIDKIT_PY_UTILS}:$PYTHONPATH

cd {UQ_RUN_ROOT}

INDICES=({_indices_str})
printf '%s\\n' "${{INDICES[@]}}" | xargs -P {WORKERS_PER_NODE} -I {{}} bash -c '
    dir="sample_runs/run_{{}}"
    solver=$(ls "$dir"/*.solver.json 2>/dev/null | head -1)
    if [[ -z "$solver" ]]; then
        echo "NO_SOLVER $dir" >> "{_slurm_dir}/resubmit_{node_i:02d}_failed.txt"
        exit 0
    fi
    cd "$dir"
    {runner} "$(basename $solver)" > stdout.txt 2> stderr.txt \\
        || echo "FAILED $dir" >> "{_slurm_dir}/resubmit_{node_i:02d}_failed.txt"
'
"""
            script_path = os.path.join(_slurm_dir, f"resubmit_{node_i:02d}.sh")
            with open(script_path, "w") as f:
                f.write(resubmit_script)
            os.chmod(script_path, 0o755)

            # Submit resubmit script
            cmd = ["sbatch", script_path]
            print(" ".join(cmd))
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                jid = result.stdout.strip().split()[-1]
                _resubmit_job_ids.append(jid)
                print(f"  submitted job {jid}  ({len(_chunk_indices)} runs)")
            else:
                print(f"  ERROR: {result.stderr.strip()}")

        # Save resubmit job IDs
        _resubmit_metadata = {
            "resubmit_job_ids": _resubmit_job_ids,
            "resubmit_indices": _resubmit_indices,
            "timestamp": datetime.datetime.now().isoformat(),
        }
        _resubmit_job_ids_file = os.path.join(_slurm_dir, "resubmit_job_ids.txt")
        import json as json_module

        with open(_resubmit_job_ids_file, "w") as f:
            json_module.dump(_resubmit_metadata, f, indent=2)

        print(f"\nSubmitted {len(_resubmit_job_ids)} resubmit jobs")
        print(f"Saved to: {_resubmit_job_ids_file}")
        print(f"Run next cell: 'Check Resubmit Status' to monitor progress.")

In [ ]:
# === Check Resubmit Status ===
# Monitor progress of resubmitted failed runs.

import json as json_module

_resubmit_job_ids_file = os.path.join(UQ_RUN_ROOT, "slurm", "resubmit_job_ids.txt")
if not os.path.exists(_resubmit_job_ids_file):
    print(f"No resubmit job file found at {_resubmit_job_ids_file}")
    print("Run 'Resubmit Failed Runs' first.")
else:
    with open(_resubmit_job_ids_file, "r") as f:
        _resubmit_metadata = json_module.load(f)

    _resubmit_job_ids = _resubmit_metadata.get("resubmit_job_ids", [])
    _resubmit_indices = _resubmit_metadata.get("resubmit_indices", [])

    print(f"Resubmitted: {_resubmit_metadata.get('timestamp')}")
    print(
        f"Resubmitting {len(_resubmit_indices)} runs with {len(_resubmit_job_ids)} SLURM jobs"
    )
    print()

    if _resubmit_job_ids:
        cmd = [
            "sacct",
            "-j",
            ",".join(_resubmit_job_ids),
            "--format=JobID,JobName,State,Elapsed,ExitCode",
        ]
        print(" ".join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)

        # Parse status
        lines = result.stdout.strip().split("\n")[1:]
        completed = failed = running = pending = 0
        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 3:
                state = parts[2]
                if state == "COMPLETED":
                    completed += 1
                elif state in ("FAILED", "TIMEOUT", "CANCELLED"):
                    failed += 1
                elif state == "RUNNING":
                    running += 1
                elif state in ("PENDING", "CONFIGURING"):
                    pending += 1

        print(
            f"\nResubmit Status: {completed} COMPLETED, {running} RUNNING, {pending} PENDING, {failed} FAILED"
        )

        if failed > 0:
            print(
                f"\n⚠ {failed} resubmit jobs still failed. Inspect logs or resubmit again."
            )
        elif running + pending > 0:
            print(
                f"\n⏳ {running + pending} jobs still in progress. Re-run this cell to check status."
            )
        else:
            print(f"\n✓ All resubmit jobs completed successfully!")

In [10]:
# === Check job status ===
# Re-run this cell to poll until all jobs complete.
# Once all show COMPLETED with no failures, proceed to the collect cell.

# --- sacct status (requires jobs to have run) ---
_all_ids = list(_job_ids)
if COLLECT_IN_SLURM and "_collect_job_id" in dir() and _collect_job_id:
    _all_ids.append(_collect_job_id)

if _all_ids:
    r = subprocess.run(
        [
            "sacct",
            "-j",
            ",".join(_all_ids),
            "--format=JobID,JobName,State,Elapsed,ExitCode",
            "--noheader",
        ],
        capture_output=True,
        text=True,
    )
    print("sacct output:")
    print(r.stdout or "  (no output yet -- jobs may still be pending)")

# --- scan failed.txt files from each chunk ---
print("\nFailed runs (from chunk *_failed.txt files):")
_all_failed = []
for node_i in range(N_NODES):
    fpath = os.path.join(_slurm_dir, f"chunk_{node_i:02d}_failed.txt")
    if os.path.exists(fpath):
        lines = open(fpath).read().strip().splitlines()
        if lines:
            _all_failed.extend(lines)
            print(f"  chunk {node_i:02d}: {lines}")
        else:
            print(f"  chunk {node_i:02d}: OK (empty failed file)")
    else:
        print(f"  chunk {node_i:02d}: no failed.txt yet (pending or running)")

# --- check if collection completed in SLURM ---
_collect_completed = False
_collect_failed = False
if COLLECT_IN_SLURM and "_collect_job_id" in dir() and _collect_job_id:
    r_collect = subprocess.run(
        ["sacct", "-j", _collect_job_id, "--format=State,ExitCode", "--noheader"],
        capture_output=True,
        text=True,
    )
    if r_collect.stdout:
        _state_line = r_collect.stdout.strip().split("\n")[
            0
        ]  # Get first line (main job, not substeps)
        _state_parts = _state_line.split()
        _collect_state = _state_parts[0] if _state_parts else ""
        _collect_exit_code = _state_parts[1] if len(_state_parts) > 1 else ""

        if "COMPLETED" in _collect_state:
            _collect_completed = True
            _collect_failed = "0:0" not in _collect_exit_code

            if _collect_failed:
                print(
                    f"\n⚠️  Collection job {_collect_job_id} COMPLETED but with non-zero exit code: {_collect_exit_code}"
                )
            else:
                print(
                    f"\n✅ Collection in SLURM successful (job {_collect_job_id} COMPLETED with exit code 0:0)"
                )
                print(
                    "   You can skip the collection cell and proceed directly to results cells."
                )

_still_running = "RUNNING" in r.stdout or "PENDING" in r.stdout
if _still_running:
    print(
        "\nJobs still in progress (RUNNING or PENDING). Re-run this cell to poll again."
    )
elif _collect_completed and not _collect_failed:
    print("\nAll jobs and collection completed successfully. Proceed to results cells.")
elif not _all_failed and not _collect_completed:
    print(
        "\nAll simulation jobs completed. No failures detected. Safe to proceed with collect cell."
    )
elif not _all_failed and _collect_completed and _collect_failed:
    print(
        f"\nSimulation jobs completed with no failures, but collection job failed. Check logs and re-run collection cell."
    )
else:
    print(
        f"\nAll jobs completed. {len(_all_failed)} failed run(s) -- investigate before collecting."
    )

NameError: name '_job_ids' is not defined

## collect results

If `COLLECT_IN_SLURM=True`, the collect job was submitted automatically and will run after all sim jobs complete — skip the collect cell below and go straight to **results size** once the collect job shows COMPLETED in the status cell.

If `COLLECT_IN_SLURM=False` (default), run the collect cell below after all simulations complete.

**Note on threading:** `collect_parallel()` uses a `ThreadPoolExecutor`. The GIL is released during file I/O, so threads genuinely overlap on Lustre reads/writes. Bottleneck is Lustre MDS metadata throughput, not CPU. Safe range on Kestrel: 32 (conservative) to 64 (aggressive).


In [ ]:
# === Collect results -> Parquet ===
#
# SERIALIZE_MODE options (set in config cell above):
#   "stacked"  - all runs in one results.parquet; good for N<500, single-df analysis
#   "per_run"  - one run_NNNN.parquet per run under results/; good for large N (1000+)
#
# For per_run mode, uses collect_parallel() from gridkit_utils -- ThreadPoolExecutor,
# I/O-bound, GIL released during file reads/writes; bottleneck is Lustre MDS, not CPU.
# Kestrel Lustre safe range: 32 (conservative) to 64 (aggressive); above 64 rarely helps.
COLLECT_WORKERS = 32

if SERIALIZE_MODE == "stacked":
    result = collect_and_save(
        UQ_RUN_ROOT,
        samples_df,
        UQ_OUT_PATH,
        mode=SERIALIZE_MODE,
        n_samples=N_SAMPLES,
    )
    results_df = result
    print(f"results_df: {results_df.shape}")
    results_df
else:
    from gridkit_utils import collect_parallel

    result, _missing = collect_parallel(
        UQ_RUN_ROOT,
        UQ_OUT_PATH,
        n_workers=COLLECT_WORKERS,
        n_samples=N_SAMPLES,
    )
    for p in result[:5]:
        print(f"  {p}")
    if len(result) > 5:
        print(f"  ... ({len(result) - 5} more)")

  500/1000 processed, 500 written, 0 missing
  1000/1000 processed, 1000 written, 0 missing
Done. Written 1000 parquet files to /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs
  /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs/run_000.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs/run_001.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs/run_002.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs/run_003.parquet
  /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-ian-csv/runs/run_004.parquet
  ... (995 more)


## results size + data quality

Reports disk/memory footprint and spot-checks a sample of runs for missing files, NaNs, column consistency, and soft physical bounds.


In [11]:
# === Results size: in-memory and on-disk ===
if SERIALIZE_MODE == "stacked":
    mem_mb = results_df.memory_usage(deep=True).sum() / 1024**2
    disk_mb = os.path.getsize(UQ_OUT_PATH) / 1024**2
    print(f"results_df:  {results_df.shape[0]:,} rows x {results_df.shape[1]} cols")
    print(f"  in-memory: {mem_mb:.1f} MB")
    print(f"  parquet on disk: {disk_mb:.2f} MB")
else:
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    total_disk_mb = sum(os.path.getsize(f) for f in run_files) / 1024**2
    print(f"per_run: {len(run_files)} files in {UQ_OUT_PATH}")
    if run_files:
        print(
            f"  total on disk: {total_disk_mb:.2f} MB  ({total_disk_mb/len(run_files):.2f} MB/run)"
        )

# === Data quality check ===
# Spot-checks QC_SAMPLE_N randomly chosen runs for completeness, NaNs,
# column consistency, and soft physical bounds (warnings only, not failures).
QC_SAMPLE_N = 100

# Soft bounds -- wide enough to allow fault transients, just catches diverged/corrupt runs.
# Note: omega is speed *deviation* (delta-omega in pu), not absolute speed.
#       Steady-state value is ~0; small excursions either side are normal.
QC_BOUNDS = {
    "Vm": (0.0, 1.6),  # bus voltage magnitude (pu)
    "Va": (-4.0, 4.0),  # bus voltage angle (rad)
    "omega": (-1.0, 1.0),  # generator speed deviation (pu); steady-state ~0
    "delta": (-20.0, 20.0),  # generator angle (rad)
}

if SERIALIZE_MODE == "per_run":
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    n_files = len(run_files)
    print(f"\n--- data quality ({min(QC_SAMPLE_N, n_files)} sampled runs) ---")
    print(
        f"Files:    {n_files} / {N_SAMPLES}  ({'OK' if n_files == N_SAMPLES else 'MISSING ' + str(N_SAMPLES - n_files)})"
    )

    rng_qc = np.random.default_rng(0)  # fixed seed: same 100 runs every re-run
    # rng_qc = np.random.default_rng()  # no seed: new random selection every re-run
    sample_paths = [
        run_files[i]
        for i in rng_qc.choice(n_files, size=min(QC_SAMPLE_N, n_files), replace=False)
    ]

    nan_runs, bounds_violations, col_counts = [], [], set()
    for fpath in sample_paths:
        run_id = os.path.basename(fpath).replace("run_", "").replace(".parquet", "")
        df = pd.read_parquet(fpath)
        col_counts.add(len(df.columns))

        if df.isnull().any().any():
            nan_runs.append((run_id, df.columns[df.isnull().any()].tolist()))

        for sig, (lo, hi) in QC_BOUNDS.items():
            for col in [c for c in df.columns if c.endswith(f"_{sig}") or c == sig]:
                vmin, vmax = df[col].min(), df[col].max()
                if vmin < lo or vmax > hi:
                    bounds_violations.append(
                        (run_id, col, f"{vmin:.3f}..{vmax:.3f}", f"[{lo}, {hi}]")
                    )

    if len(col_counts) == 1:
        print(f"Columns:  {col_counts.pop()} per file (consistent)")
    else:
        print(f"Columns:  INCONSISTENT: {col_counts}")

    if nan_runs:
        print(f"NaNs:     {len(nan_runs)} run(s) with NaNs")
        for run_id, cols in nan_runs[:5]:
            print(f"  run_{run_id}: {cols}")
        if len(nan_runs) > 5:
            print(f"  ... ({len(nan_runs) - 5} more)")
    else:
        print(f"NaNs:     none")

    if bounds_violations:
        print(f"Bounds:   {len(bounds_violations)} soft violation(s)")
        for run_id, col, rng_str, expected in bounds_violations[:10]:
            print(f"  run_{run_id}  {col}  {rng_str}  expected {expected}")
        if len(bounds_violations) > 10:
            print(f"  ... ({len(bounds_violations) - 10} more)")
    else:
        print(f"Bounds:   all within expected ranges")

per_run: 1025 files in /kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results
  total on disk: 10876.26 MB  (10.61 MB/run)

--- data quality (100 sampled runs) ---
Files:    1025 / 1025  (OK)
Columns:  481 per file (consistent)
NaNs:     none
Bounds:   all within expected ranges


## sanity check plot
Loads a small subset of runs and plots one signal per (element, variable).
For full ensemble visualization use `gridkit_viv.ipynb`.

In [12]:
# === Sanity check plot: 2 runs x 1 bus signal + 1 gen signal per var ===
PLOT_MAX_RUNS = 5
PLOT_MAX_BUS_N = 2
PLOT_MAX_GEN_N = 2

from collections import defaultdict
from gridkit_utils import MONITORABLE_VARS_BY_ELEMENT

# rng_plot = np.random.default_rng(SEED)  # fixed seed: same selection every run
rng_plot = np.random.default_rng()  # no seed: new random selection every run

# Load subset depending on serialize mode
if SERIALIZE_MODE == "stacked":
    _plot_df_full = results_df
else:
    run_files = sorted(glob.glob(os.path.join(UQ_OUT_PATH, "run_*.parquet")))
    if not run_files:
        raise FileNotFoundError(f"No run_*.parquet files found in {UQ_OUT_PATH}")
    chosen_idx = sorted(
        rng_plot.choice(
            len(run_files), size=min(PLOT_MAX_RUNS, len(run_files)), replace=False
        )
    )
    frames = []
    for idx in chosen_idx:
        fpath = run_files[idx]
        run_id = int(
            os.path.basename(fpath).replace("run_", "").replace(".parquet", "")
        )
        df = pd.read_parquet(fpath)
        df.insert(0, "run_id", run_id)
        frames.append(df)
    _plot_df_full = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(chosen_idx)} run files: {[run_files[i] for i in chosen_idx]}")

param_cols = list(samples_df.columns)
skip_cols = {"run_id", "time", "Solver Status"} | set(param_cols)
mon_cols = [c for c in _plot_df_full.columns if c not in skip_cols]

all_run_ids = sorted(_plot_df_full["run_id"].unique())
plot_run_ids = list(
    rng_plot.choice(
        all_run_ids, size=min(PLOT_MAX_RUNS, len(all_run_ids)), replace=False
    )
)
print(f"Runs: {plot_run_ids}")
plot_df = _plot_df_full[_plot_df_full["run_id"].isin(plot_run_ids)]

col_groups = defaultdict(list)
for col in mon_cols:
    parts = col.split("_")
    key = ("Bus", parts[-1]) if parts[0] == "Bus" else (parts[0], parts[-1])
    col_groups[key].append(col)

for (elem, var), cols in sorted(col_groups.items()):
    max_n = PLOT_MAX_BUS_N if elem == "Bus" else PLOT_MAX_GEN_N
    plot_cols = list(rng_plot.choice(cols, size=min(max_n, len(cols)), replace=False))
    melted = plot_df[["time", "run_id"] + plot_cols].melt(
        id_vars=["time", "run_id"], var_name="signal", value_name=var
    )
    fig = px.line(
        melted,
        x="time",
        y=var,
        color="signal",
        line_group="run_id",
        title=f"{elem} {var} - {plot_cols}, {len(plot_run_ids)} runs",
        labels={"time": "Time (s)", var: var},
    )
    _ = fig.update_traces(opacity=0.7)
    _ = fig.update_layout(legend_title="signal")
    _ = fig.show()

Loaded 5 run files: ['/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results/run_0208.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results/run_0688.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results/run_0771.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results/run_0819.parquet', '/kfs2/projects/scidac/scidac-data/gridkit-runs/illinois-ian-csv/results/run_0831.parquet']
Runs: [np.int64(208), np.int64(831), np.int64(771), np.int64(688), np.int64(819)]


## samples sanity check
Plot sampled H values for a chosen generator, with nominal and ±1 std reference lines.

In [ ]:
# === Samples sanity check: H distribution for one generator ===
# Set to any id that appears in PARAM_SPECS, e.g. "2_1" or "23_1"
PLOT_GEN_ID = "2_1"

# --- find the spec for the chosen generator ---
_spec = next((s for s in PARAM_SPECS if s["id"] == PLOT_GEN_ID), None)
if _spec is None:
    raise ValueError(
        f"Generator id '{PLOT_GEN_ID}' not found in PARAM_SPECS. "
        f"Available: {[s['id'] for s in PARAM_SPECS]}"
    )

_col = f"Genrou_{PLOT_GEN_ID}_H"
if _col not in samples_df.columns:
    # column name may vary; fall back to first matching column
    _col = next((c for c in samples_df.columns if PLOT_GEN_ID in c), None)
    if _col is None:
        raise KeyError(
            f"No column for gen '{PLOT_GEN_ID}' in samples_df. "
            f"Columns: {list(samples_df.columns)}"
        )

_vals = samples_df[_col].values

if _spec["dist"] == "normal":
    _nominal = _spec["mean"]
    _std = _spec["std"]
    _dist_label = f"N({_nominal:.3f}, {_std:.4f})"
else:
    _nominal = _spec["nominal"]
    _std = _nominal * _spec["pct"]
    _dist_label = f"U({_nominal*(1-_spec['pct']):.4f}, {_nominal*(1+_spec['pct']):.4f})"

_plot_df = pd.DataFrame({"sample_index": range(len(_vals)), "H": _vals})

fig = px.scatter(
    _plot_df,
    x="sample_index",
    y="H",
    opacity=0.5,
    title=f"Sampled H values — Genrou {PLOT_GEN_ID}  ({_dist_label},  N={len(_vals)})",
    labels={"sample_index": "Sample index", "H": "H (inertia constant)"},
)
_ = fig.add_hline(
    y=_nominal,
    line_dash="solid",
    line_color="black",
    annotation_text=f"nominal = {_nominal:.4f}",
    annotation_position="top right",
)
_ = fig.add_hline(
    y=_nominal + _std,
    line_dash="dash",
    line_color="crimson",
    annotation_text=f"+1σ = {_nominal+_std:.4f}",
    annotation_position="top right",
)
_ = fig.add_hline(
    y=_nominal - _std,
    line_dash="dash",
    line_color="steelblue",
    annotation_text=f"-1σ = {_nominal-_std:.4f}",
    annotation_position="bottom right",
)
_ = fig.update_layout(showlegend=False)
_ = fig.show()

print(f"  mean:   {_vals.mean():.5f}  (nominal {_nominal:.4f})")
print(f"  std:    {_vals.std():.5f}  (target {_std:.4f})")
print(f"  min:    {_vals.min():.5f}")
print(f"  max:    {_vals.max():.5f}")

In [ ]:
# # === Samples histogram: all generators -- sanity check Gaussian shape ===
# from scipy.stats import norm as _scipy_norm
# import plotly.graph_objects as go

# for _s in PARAM_SPECS:
#     _gid = _s["id"]
#     _col = f"Genrou_{_gid}_H"
#     if _col not in samples_df.columns:
#         _col = next((c for c in samples_df.columns if _gid in c), None)
#     if _col is None:
#         print(f"  WARNING: no column found for gen {_gid}, skipping")
#         continue

#     _v = samples_df[_col].values
#     _mu = _s["mean"] if _s["dist"] == "normal" else _s["nominal"]
#     _sig = _s["std"] if _s["dist"] == "normal" else _s["nominal"] * _s["pct"]

#     # x range for the reference PDF
#     _x = np.linspace(_v.min(), _v.max(), 300)
#     _pdf = _scipy_norm.pdf(_x, loc=_mu, scale=_sig)

#     fig = go.Figure()
#     _ = fig.add_trace(
#         go.Histogram(
#             x=_v,
#             histnorm="probability density",
#             name="samples",
#             marker_color="steelblue",
#             opacity=0.7,
#             nbinsx=50,
#         )
#     )
#     _ = fig.add_trace(
#         go.Scatter(
#             x=_x,
#             y=_pdf,
#             mode="lines",
#             name=f"N({_mu:.3f}, {_sig:.4f})",
#             line=dict(color="crimson", width=2),
#         )
#     )
#     _ = fig.update_layout(
#         title=f"H samples — Genrou {_gid}  (N={len(_v)}, std={_sig:.4f} = {_sig/_mu*100:.0f}% of nominal)",
#         xaxis_title="H (inertia constant)",
#         yaxis_title="Probability density",
#         barmode="overlay",
#         legend_title="",
#     )
#     _ = fig.show()

## standalone QC

Runs table re-scan and per-run quality check. These cells are self-contained: only the imports cell is required, no setup cell needed.

1. **Runs table** — re-scans all `meta.yml` files under `gridkit-runs/` and shows completion status for every run.
2. **QC config** — set `QC_RUN_NAME` to the run you want to inspect. Reads `meta.yml` from disk and prints run parameters.
3. **QC check** — file count, NaN scan, soft bounds check on a random sample of parquet files, disk footprint, and SLURM `*_failed.txt` scan.

In [20]:
# === Existing UQ runs (re-scan) ===
# Re-run this cell to see all available runs and their completion status.
# Then set QC_RUN_NAME in the cell below.

_RUNS_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs"

_rows = []
for _meta_path in sorted(glob.glob(os.path.join(_RUNS_ROOT, "*/meta.yml"))):
    _run_name = os.path.basename(os.path.dirname(_meta_path))
    try:
        with open(_meta_path) as _f:
            _m = yaml.safe_load(_f)
    except Exception as _e:
        _rows.append({"run": _run_name, "error": str(_e)})
        continue

    _s = _m.get("sampling", {})
    _params = _s.get("params", [])
    _genrou_ids = ", ".join(p.get("id", "?") for p in _params)
    _serialize = _m.get("serialize_mode", "unknown")
    _run_dir = os.path.dirname(_meta_path)
    _n_total = _s.get("n_samples", None)

    if _serialize == "per_run":
        _runs_path = os.path.join(_run_dir, "runs")
        _n_done = (
            len(glob.glob(os.path.join(_runs_path, "run_*.parquet")))
            if os.path.isdir(_runs_path)
            else None
        )
    elif _serialize == "stacked":
        _n_done = (
            _n_total
            if os.path.exists(os.path.join(_run_dir, "results.parquet"))
            else None
        )
    else:
        _n_done = None

    _pct = (
        f"{_n_done / _n_total * 100:.0f}%" if _n_done is not None and _n_total else "NA"
    )
    _rows.append(
        {
            "run": _run_name,
            "case": _m.get("case", "?"),
            "created": _m.get("created", "?")[:10],
            "n_samples": int(_n_total) if _n_total is not None else "NA",
            "n_parquets": int(_n_done) if _n_done is not None else "NA",
            "complete": _pct,
            "dist": _s.get("dist_type", "?"),
            "serialize": _serialize,
            "Genrou_H_ids": _genrou_ids,
        }
    )

pd.set_option("display.max_rows", 30)
pd.DataFrame(_rows).sort_values("run", ascending=True).reset_index(drop=True)
pd.set_option("display.max_rows", 10)

,run,case,created,n_samples,n_parquets,complete,dist,serialize,Genrou_H_ids
0,hawaii-ian-csv,Hawaii,2026-08-11,1000,1000,100%,from_csv,per_run,"2_1, 2_2, 2_3, 2_4, 23_1, 23_10, 23_2, 23_3, 2..."
1,hawaii-v1,Hawaii,2026-06-11,NA,NA,NA,?,unknown,
2,hawaii-v10,Hawaii,2026-08-11,4000,4000,100%,uniform,per_run,"35_4, 23_9, 26_1, 27_2"
3,hawaii-v2,Hawaii,2026-06-11,NA,1000,NA,?,per_run,
4,hawaii-v3,Hawaii,2026-07-21,1000,1000,100%,normal,per_run,"2_1, 23_1, 34_1, 35_1"
5,hawaii-v4,Hawaii,2026-07-21,4000,4000,100%,normal,per_run,"2_1, 23_1"
6,hawaii-v5,Hawaii,2026-07-22,16000,16000,100%,normal,per_run,"2_1, 23_1, 34_1, 35_1"
7,hawaii-v6,Hawaii,2026-08-08,4000,4000,100%,uniform,per_run,"37_3, 26_2, 27_1, 33_1"
8,hawaii-v7,Hawaii,2026-08-08,4000,4000,100%,uniform,per_run,"27_1, 28_1, 33_1, 34_1"
9,hawaii-v8,Hawaii,2026-08-08,4000,4000,100%,uniform,per_run,"2_1, 36_1, 34_1, 26_2"


In [17]:
# === Standalone QC config ===
# Set to the run name you want to inspect (must match a directory under gridkit-runs/).
QC_RUN_NAME = "hawaii-v9"

_RUNS_ROOT = "/kfs2/projects/scidac/scidac-data/gridkit-runs"
QC_RUN_DIR = os.path.join(_RUNS_ROOT, QC_RUN_NAME)

# Load meta.yml to get all run parameters without needing the setup cell.
_meta_path = os.path.join(QC_RUN_DIR, "meta.yml")
if not os.path.exists(_meta_path):
    raise FileNotFoundError(
        f"meta.yml not found: {_meta_path}\nRun the setup cell first to write it."
    )

with open(_meta_path) as _f:
    _meta = yaml.safe_load(_f)

_qc_serialize = _meta.get("serialize_mode", "per_run")
_qc_n_total = _meta["sampling"]["n_samples"]
_qc_out_path = (
    os.path.join(QC_RUN_DIR, "results.parquet")
    if _qc_serialize == "stacked"
    else os.path.join(QC_RUN_DIR, "runs")
)

print(f"run:        {QC_RUN_NAME}")
print(f"case:       {_meta.get('case', '?')}")
print(f"created:    {_meta.get('created', '?')}")
print(f"n_samples:  {_qc_n_total}")
print(f"serialize:  {_qc_serialize}")
print(f"out_path:   {_qc_out_path}")
_params = _meta["sampling"].get("params", [])
print(f"params ({len(_params)}):")
for _p in _params:
    if _p.get("dist") == "uniform":
        print(
            f"  {_p['id']:8s}: H uniform [{_p['lo']:.4f}, {_p['hi']:.4f}]  (nominal={_p['nominal']:.4f}, pct={_p['pct']:.0%})"
        )
    else:
        print(
            f"  {_p['id']:8s}: H normal  mean={_p['mean']:.4f}, std={_p['std']:.4f} ({_p.get('std_pct', 0)*100:.0f}%)"
        )

run:        hawaii-v9
case:       Hawaii
created:    2026-08-08T12:28:55
n_samples:  4000
serialize:  per_run
out_path:   /kfs2/projects/scidac/scidac-data/gridkit-runs/hawaii-v9/runs
params (4):
  35_8    : H uniform [4.1760, 6.2640]  (nominal=5.2200, pct=20%)
  23_1    : H uniform [4.9200, 7.3800]  (nominal=6.1500, pct=20%)
  37_3    : H uniform [1.9840, 2.9760]  (nominal=2.4800, pct=20%)
  28_2    : H uniform [2.4000, 3.6000]  (nominal=3.0000, pct=20%)


In [18]:
# === Standalone QC: file count + NaN + soft bounds check ===
# Requires: QC_RUN_NAME config cell above has been run.

QC_SAMPLE_N = 100  # number of parquet files to spot-check

QC_BOUNDS = {
    "Vm": (0.0, 1.6),
    "Va": (-4.0, 4.0),
    "omega": (-1.0, 1.0),
    "delta": (-20.0, 20.0),
}

if _qc_serialize == "per_run":
    _run_files = sorted(glob.glob(os.path.join(_qc_out_path, "run_*.parquet")))
    _n_done = len(_run_files)
    print(
        f"Files:    {_n_done} / {_qc_n_total}  ({'OK' if _n_done == _qc_n_total else 'MISSING ' + str(_qc_n_total - _n_done)})"
    )

    if _n_done == 0:
        print("No parquet files found -- collection not yet run.")
    else:
        _rng = np.random.default_rng(0)
        _sample_paths = [
            _run_files[i]
            for i in _rng.choice(_n_done, size=min(QC_SAMPLE_N, _n_done), replace=False)
        ]
        _nan_runs, _bounds_violations, _col_counts = [], [], set()
        for _fpath in _sample_paths:
            _run_id = (
                os.path.basename(_fpath).replace("run_", "").replace(".parquet", "")
            )
            _df = pd.read_parquet(_fpath)
            _col_counts.add(len(_df.columns))
            if _df.isnull().any().any():
                _nan_runs.append((_run_id, _df.columns[_df.isnull().any()].tolist()))
            for _sig, (_lo, _hi) in QC_BOUNDS.items():
                for _col in [
                    c for c in _df.columns if c.endswith(f"_{_sig}") or c == _sig
                ]:
                    _vmin, _vmax = _df[_col].min(), _df[_col].max()
                    if _vmin < _lo or _vmax > _hi:
                        _bounds_violations.append(
                            (
                                _run_id,
                                _col,
                                f"{_vmin:.3f}..{_vmax:.3f}",
                                f"[{_lo}, {_hi}]",
                            )
                        )

        print(
            f"Columns:  {_col_counts.pop() if len(_col_counts) == 1 else 'INCONSISTENT: ' + str(_col_counts)} per file"
        )
        if _nan_runs:
            print(f"NaNs:     {len(_nan_runs)} run(s) with NaNs")
            for _rid, _cols in _nan_runs[:5]:
                print(f"  run_{_rid}: {_cols}")
        else:
            print(f"NaNs:     none")
        if _bounds_violations:
            print(f"Bounds:   {len(_bounds_violations)} soft violation(s)")
            for _rid, _col, _rng_str, _exp in _bounds_violations[:10]:
                print(f"  run_{_rid}  {_col}  {_rng_str}  expected {_exp}")
            if len(_bounds_violations) > 10:
                print(f"  ... ({len(_bounds_violations) - 10} more)")
        else:
            print(f"Bounds:   all within expected ranges")

        # --- disk footprint ---
        _total_mb = sum(os.path.getsize(f) for f in _run_files) / 1024**2
        print(
            f"\nDisk:     {_total_mb:.1f} MB total  ({_total_mb / _n_done:.2f} MB/run)"
        )

elif _qc_serialize == "stacked":
    if os.path.exists(_qc_out_path):
        _df = pd.read_parquet(_qc_out_path)
        print(f"stacked:  {_df.shape[0]:,} rows x {_df.shape[1]} cols")
        print(f"Disk:     {os.path.getsize(_qc_out_path) / 1024**2:.2f} MB")
        print(f"NaNs:     {_df.isnull().sum().sum()}")
    else:
        print(f"results.parquet not found: {_qc_out_path}")

# --- SLURM failed.txt scan ---
_slurm_dir = os.path.join(QC_RUN_DIR, "slurm")
if os.path.isdir(_slurm_dir):
    _failed_files = sorted(glob.glob(os.path.join(_slurm_dir, "*_failed.txt")))
    _all_failed = []
    for _fp in _failed_files:
        _lines = open(_fp).read().strip().splitlines()
        _all_failed.extend(_lines)
    if _all_failed:
        print(f"\nSLURM failed runs ({len(_all_failed)}):")
        for _line in _all_failed[:20]:
            print(f"  {_line}")
    else:
        print(f"\nSLURM:    no failed runs in slurm/*_failed.txt")
else:
    print(
        f"\nSLURM:    no slurm/ dir found (option A serial run, or not yet submitted)"
    )

Files:    4000 / 4000  (OK)
Columns:  153 per file
NaNs:     none
Bounds:   all within expected ranges

Disk:     13475.4 MB total  (3.37 MB/run)

SLURM:    no failed runs in slurm/*_failed.txt


# end